## 4.1 专家迭代
在更早的强化学习文献里，**Expert Iteration (ExIt)** 讲的是“搜索器先做出更强决策，神经网络再去模仿这些更强决策”；到了语言模型里，更接近这节代码精神的，是 **STaR**：模型先自己生成推理链，只保留那些最终答对的样本，再用这些“自己挖出来的正确推理”继续微调自己。
如果说：

* **SFT** 是把外部专家轨迹塞给模型；
* **PPO / GRPO** 是直接拿 reward 在线推策略；

那 **EI** 刚好夹在中间：
它不像纯 SFT 那样完全依赖外部老师，也不像 RL 那样直接优化期望奖励；它更像一种**自举式的数据制造机**——模型先自己采样，再由 verifier 决定哪些样本有资格晋升为“专家数据”，最后还是回到熟悉的 **SFT loss** 去学。这个逻辑和 STaR 的核心循环非常像：生成 rationales，只保留最终答对的那些，再反复 fine-tune。

### 4.1.1 一个还不够强的模型，凭什么能教会自己
答案不是“它天生很聪明”，而是：
**它在随机采样下，偶尔会走出正确路径。**

只要有这三样东西同时存在，EI 就能运转：

1. **探索性**：同一道题，多采几次，路径会不一样
2. **可验证性**：我们能自动判断这次到底对没对
3. **可再利用性**：一旦某条路径被证实是对的，就能回灌成训练数据

模型不需要一开始就会稳定地产生好推理，它只要“**偶尔采到正确推理**”，整个循环就有了种子。之后模型越强，生成的数据越好；数据越好，下一轮模型又更强——这是一个真正的自举闭环。

### 4.1.2 数据聚合对比
| | **DAgger** | **EI / STaR** |
|---|---|---|
| **核心思想** | Dataset Aggregation | 自我生成的专家轨迹 |
| **数据从哪来** | 让当前策略去交互，但**由外部专家**打标签 | 让当前策略去采样，**由 verifier（结果对错）** 打标签 |
| **为什么要这样做** | 解决 covariate shift（分布偏移）：训练集是专家轨迹，但运行时模型走偏了，没见过这些状态，就不会处理了。DAgger 把模型自己跑出的“烂状态”也交给专家纠正，训练数据覆盖了模型的真实访问分布。 | 解决训练数据稀缺：外部专家数据有限，模型必须从自己海量的采样中“淘金”，用 reward signal 筛选出能用的轨迹，扩展训练集。 |
| **损失函数** | 对专家动作做监督学习（分类/回归 loss） | 对正确轨迹做 SFT loss（next-token prediction） |
| **迭代形式** | 模型变强 → 再跑新数据 → 专家再标 → 再训练 | 模型变强 → 再采样新推理 → 筛选对的 → 再 SFT |

- 两者都打破了传统“一次性拿固定数据集训练”的模式，进入了一个**迭代式数据收集与训练**的循环：
- 两者都遵循**“用当前策略收集数据，用某种 oracle 提供正确标签，再监督学习”**的范式。

- **DAgger 依赖外部专家**：  
  那个打标签的 oracle 是一个**比当前策略更强的外部专家**（比如人类驾驶员、规则型控制器）。模型永远在模仿一个比自己强的对象。
- **EI / STaR 没有外部专家**：  
  那个“正确标签”不是别人教的，而是**环境反馈**（数学题答案对没对、代码跑没跑通）。模型只是在筛选自己“蒙对了”的那部分样本。

### 4.1.3 在线强化学习对比
EI 和 RL都包含：

* 采样
* 打分
* 用“高分结果”推动下一轮模型变好
但它们的**更新方式**其实不同：

* **RL（比如 PPO / GRPO）**：在线地用 reward 改 policy
* **EI**：先把 reward 变成一个硬筛选器，再对筛出来的数据做 SFT

所以从代码角度看，EI 更像：

```text
rollout
  -> verifier
  -> accepted trajectories
  -> supervised fine-tuning
```
而不是：
```text
rollout
  -> reward
  -> policy gradient / advantage-based update
```

此外，EI 完全不信任 reward 的连续数值，只信任它的二元判断：
> “这条轨迹最终答对了？留下。答错了？扔掉。”

它不关心“哪个 token 导致了正确”，只关心“整个推理链最终对不对”。
然后它对留下的轨迹做最简单的 SFT，不像 PPO 那样，没有 clipping，没有 KL，没有 advantage。

这带来了两个巨大的好处：
1. 稳定性：SFT 本身就是最稳定的更新方式。模型只是在模仿它自己成功的那些推理链，永远不会因为一次更新太猛而崩掉。
2. 简单性：你不需要维护 value network，不需要调 KL 系数，不需要担心 reward scale 变化。你只需要一个 is_correct 布尔值。

代价是：**效率低**。
因为扔掉了所有错误样本，等于浪费了大量计算。如果模型正确率只有 10%，那 90% 的采样都是白费的。

## 4.2 EI 的四段式训练循环：采样、过滤、微调、同步
一轮 Expert Iteration 可以被拆成四步：

1. Rollout：对同一道题采样多个候选解
2. Verify：用规则奖励函数筛出“格式对 + 答案对”的轨迹
3. Fine-tune：把这些轨迹当成新的 SFT 数据训练
4. Sync：把新 policy 权重同步到 vLLM，进入下一轮采样

In [ ]:
import sys
from pathlib import Path

ROOT = Path("..").absolute()
sys.path.append(str(ROOT))

import import_ipynb
import os
import numpy as np
import torch
from typing import Optional

from sft.sft import init_vllm, load_policy_into_vllm_instance, tokenize_prompt_and_output, compute_entropy, get_response_log_probs, sft_microbatch_train_step, r1_zero_reward_fn, log_generations

In [ ]:
def init_ei_experiment(
    batch_size,
    micro_batch_size,
    wandb_project,
    wandb_run_name,
    prompt_path,
    extra_config=None,
):
    """
    计算 grad_accum_steps，初始化 wandb，加载 prompt 模板。
    返回：
      grad_accum_steps, wandb_run, r1_template
    """
    if batch_size % micro_batch_size != 0:
        raise ValueError("batch_size 必须能被 micro_batch_size 整除。")

    grad_accum_steps = batch_size // micro_batch_size

    config = dict(extra_config or {})
    config["grad_accum_steps"] = grad_accum_steps

    run = wandb.init(
        project=wandb_project,
        name=wandb_run_name,
        config=config,
    )

    run.define_metric("global_step")
    run.define_metric("ei_step")
    run.define_metric("train/*", step_metric="global_step")
    run.define_metric("eval/*", step_metric="global_step")
    run.define_metric("ei/*", step_metric="ei_step")

    with open(prompt_path, "r") as f:
        r1_template = f.read().strip()

    return grad_accum_steps, run, r1_template

### 4.2.1 加载 tokenizer / policy / optimizer / vLLM

In [ ]:
def load_ei_models_and_optimizer(model_id, device, vllm_device, seed, vllm_gpu_util, lr):
    """
    加载 tokenizer, policy, optimizer, vLLM。
    """
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    policy = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.bfloat16,
        low_cpu_mem_usage=True,
        attn_implementation="flash_attention_2",
    ).to(device)

    policy.config.use_cache = False
    policy.gradient_checkpointing_enable()
    policy.train()

    optimizer = AdamW(policy.parameters(), lr=lr)
    vllm_inst = init_vllm(model_id, vllm_device, seed, vllm_gpu_util)

    return tokenizer, policy, optimizer, vllm_inst

### 4.2.2 准备 question pool
这个 cell 只做训练问题池，不做验证集。
EI 的训练数据源不是静态 prompt-response，而是问题池 + 当前模型 rollouts。

In [ ]:
def prepare_question_pool(train_data_path, prompt_template):
    """
    从训练集读取 question pool。
    返回：
      List[{"prompt": ..., "gold": ...}]
    """
    question_pool = []

    with open(train_data_path, "r") as f:
        for line in f:
            item = json.loads(line)

            question = item["question"]
            if "gold" in item:
                gold = item["gold"]
            else:
                raw_a = item["answer"]
                gold = raw_a.split("####")[-1].strip() if "####" in raw_a else raw_a.strip()

            prompt = prompt_template.replace("{question}", question)
            question_pool.append({
                "prompt": prompt,
                "gold": gold,
            })

    return question_pool

### 4.2.3 准备验证集样本

In [ ]:
def prepare_validation_samples(val_data_path, prompt_template, max_eval_samples):
    """
    加载验证集，返回：
      List[{"prompt": ..., "gold": ...}]
    """
    val_samples = []

    with open(val_data_path, "r") as f:
        for i, line in enumerate(f):
            if i >= max_eval_samples:
                break

            item = json.loads(line)
            raw_a = item["answer"]
            gold = raw_a.split("####")[-1].strip() if "####" in raw_a else raw_a.strip()

            prompt = prompt_template.replace("{question}", item["question"])
            val_samples.append({
                "prompt": prompt,
                "gold": gold,
            })

    return val_samples

### 4.2.4 构造 eval / rollout 两种 sampling params

In [ ]:
def build_sampling_params(max_tokens, rollouts):
    """
    构造 EI 需要的两套采样参数：
      1. eval_params: greedy，用于验证集评估
      2. rollout_params: n=rollouts, temperature=1.0，用于探索采样
    """
    eval_params = SamplingParams(
        temperature=0.0,
        max_tokens=max_tokens,
        stop=[ANSWER_CLOSE],
        include_stop_str_in_output=True,
    )

    rollout_params = SamplingParams(
        n=rollouts,
        temperature=1.0,
        max_tokens=max_tokens,
        min_tokens=4,
        stop=[ANSWER_CLOSE],
        include_stop_str_in_output=True,
    )

    return eval_params, rollout_params

### 4.2.5 Step 0 初始评估

In [ ]:
def run_initial_ei_evaluation(policy, vllm_inst, eval_params, val_samples):
    """
    训练开始前的 baseline 评估。
    """
    print("\n[Step 0] 执行训练前初始评估...")
    policy.eval()
    load_policy_into_vllm_instance(policy, vllm_inst)

    metrics = log_generations(
        vllm_model=vllm_inst,
        sampling_params=eval_params,
        prompts=[s["prompt"] for s in val_samples],
        ground_truths=[s["gold"] for s in val_samples],
        reward_fn=r1_zero_reward_fn,
        step=0,
        log_prefix="eval",
    )
    print(f"Initial Accuracy: {metrics.get('eval/accuracy', 0):.2%}")

    policy.train()
    return metrics

### 4.2.6 采样阶段（Rollout）

In [ ]:
def sample_rollouts_from_question_pool(question_pool, ei_batch_size, vllm_inst, rollout_params):
    """
    从 question pool 中随机抽一批题目，交给当前模型做多次 rollout。
    返回：
      batch_db, outputs
    """
    batch_db = random.sample(question_pool, min(ei_batch_size, len(question_pool)))
    print(f">> 正在对 {len(batch_db)} 个问题进行采样 (G={rollout_params.n})...")

    outputs = vllm_inst.generate(
        [q["prompt"] for q in batch_db],
        rollout_params
    )
    return batch_db, outputs

### 4.2.7 过滤阶段（Verify）
只保留 reward == 1.0 的样本，组成下一轮 SFT 数据。

In [ ]:
def filter_expert_samples(batch_db, outputs):
    """
    过滤 rollout 输出，只保留 reward == 1.0 的专家样本。
    返回：
      expert_raw_data, success_rate
    """
    expert_raw_data = []
    total_candidates = 0

    for i, output in enumerate(outputs):
        current_gold = batch_db[i]["gold"]

        for candidate in output.outputs:
            total_candidates += 1
            reward_info = r1_zero_reward_fn(candidate.text, current_gold)

            if reward_info["reward"] == 1.0:
                expert_raw_data.append({
                    "prompt": batch_db[i]["prompt"],
                    "response": candidate.text,
                })

    success_rate = len(expert_raw_data) / max(total_candidates, 1)
    return expert_raw_data, success_rate

### 4.2.8 动态计算 train_steps

In [ ]:
def compute_train_steps_from_expert_data(num_expert_samples, epochs_per_ei, batch_size):
    """
    根据当前轮获得的专家样本数，动态计算训练步数。
    """
    train_steps = (num_expert_samples * epochs_per_ei) // batch_size
    train_steps = max(1, train_steps)
    return train_steps

### 4.2.9 对当前轮专家样本做一次 tokenize
类似 DeepSeek-R1 把 rejection sampling、SFT 和 RL 串成多阶段，就是因为这几种东西不是互相排斥，而是可以各干各的活：有的负责探索，有的负责整理数据，有的负责在线搜索。

In [ ]:
def tokenize_expert_data(expert_raw_data, tokenizer):
    """
    对当前 EI 轮收集到的专家样本做 tokenize。
    """
    tokenized_expert_data = tokenize_prompt_and_output(
        prompt_strs=[ex["prompt"] for ex in expert_raw_data],
        output_strs=[ex["response"] for ex in expert_raw_data],
        tokenizer=tokenizer,
    )
    return tokenized_expert_data

### 4.2.10 单个 EI optimizer step
本质还是一次 SFT 更新，只是它训练的对象不再是静态人工数据，而是当前 EI 轮筛出来的专家样本。

In [ ]:
def train_one_ei_optimizer_step(
    policy,
    optimizer,
    tokenized_expert_data,
    micro_batch_size,
    grad_accum_steps,
    device,
    global_step,
):
    """
    执行一次 optimizer.step()。
    返回：
      avg_loss, avg_global_entropy, avg_response_entropy
    """
    policy.train()

    accumulated_loss = 0.0
    accumulated_global_entropy = 0.0
    accumulated_response_entropy = 0.0

    for _ in range(grad_accum_steps):
        batch = get_batch(tokenized_expert_data, micro_batch_size, device)

        response_outputs = get_response_log_probs(
            model=policy,
            input_ids=batch["input_ids"],
            labels=batch["labels"],
            attention_mask=batch["attention_mask"],
            return_token_entropy=True,
        )

        log_probs = response_outputs["log_probs"]
        token_entropy = response_outputs["token_entropy"]

        with torch.no_grad():
            valid_token_mask = batch["valid_mask"].bool()
            current_res_mask = batch["response_mask"].bool() & valid_token_mask

            avg_res_ent = (
                token_entropy[current_res_mask].mean().item()
                if current_res_mask.any() else 0.0
            )
            avg_glob_ent = (
                token_entropy[valid_token_mask].mean().item()
                if valid_token_mask.any() else 0.0
            )

            normalize_constant = current_res_mask.sum().clamp_min(1).item()

        loss, _ = sft_microbatch_train_step(
            policy_log_probs=log_probs,
            response_mask=current_res_mask,
            gradient_accumulation_steps=grad_accum_steps,
            normalize_constant=normalize_constant,
        )
        loss.backward()

        accumulated_loss += loss.item() * grad_accum_steps
        accumulated_global_entropy += avg_glob_ent
        accumulated_response_entropy += avg_res_ent

    torch.nn.utils.clip_grad_norm_(policy.parameters(), 1.0)
    optimizer.step()
    optimizer.zero_grad(set_to_none=True)

    avg_loss = accumulated_loss / grad_accum_steps
    avg_global_entropy = accumulated_global_entropy / grad_accum_steps
    avg_response_entropy = accumulated_response_entropy / grad_accum_steps

    wandb.log({
        "train/loss": avg_loss,
        "train/global_entropy": avg_global_entropy,
        "train/response_entropy": avg_response_entropy,
        "global_step": global_step,
    })

    return avg_loss, avg_global_entropy, avg_response_entropy

### 4.2.11 训练中途评估
（按 step 频率触发）

In [ ]:
def run_mid_training_eval_if_needed(
    policy,
    vllm_inst,
    eval_params,
    val_samples,
    global_optim_step,
    eval_every_steps,
):
    """
    如果达到评估步数，则执行一次中途评估。
    """
    if global_optim_step % eval_every_steps != 0:
        return None

    print(f"\n[Step {global_optim_step}] 训练中途评估...")
    policy.eval()
    load_policy_into_vllm_instance(policy, vllm_inst)

    metrics = log_generations(
        vllm_model=vllm_inst,
        sampling_params=eval_params,
        prompts=[s["prompt"] for s in val_samples],
        ground_truths=[s["gold"] for s in val_samples],
        reward_fn=r1_zero_reward_fn,
        step=global_optim_step,
        log_prefix="eval",
    )

    policy.train()
    return metrics

### 4.2.12 每轮 EI 结束后的评估

In [ ]:
def run_end_of_iteration_eval(policy, vllm_inst, eval_params, val_samples, global_optim_step):
    """
    当前 EI 轮训练完成后，做一次评估。
    """
    print(">> 正在进行本轮迭代的验证...")
    policy.eval()
    load_policy_into_vllm_instance(policy, vllm_inst)

    metrics = log_generations(
        vllm_model=vllm_inst,
        sampling_params=eval_params,
        prompts=[s["prompt"] for s in val_samples],
        ground_truths=[s["gold"] for s in val_samples],
        reward_fn=r1_zero_reward_fn,
        step=global_optim_step,
        log_prefix="eval",
    )
    print(f"Accuracy: {metrics.get('eval/accuracy', 0):.2%}")

    policy.train()
    return metrics

### 4.2.13 每轮 EI 保存一次 checkpoint

In [ ]:
def save_ei_checkpoint(policy, tokenizer, output_dir, ei_step, global_optim_step):
    """
    每轮 EI 结束后保存一次 checkpoint。
    """
    save_path = os.path.join(output_dir, f"ei_iter{ei_step}_step{global_optim_step}")
    os.makedirs(save_path, exist_ok=True)

    policy.save_pretrained(save_path)
    tokenizer.save_pretrained(save_path)

    print(f"Checkpoint saved to {save_path}")
    return save_path

### 4.2.14 训练主流程
- G（每题的 rollouts 数量）→ 买“探索深度”
    - 让同一题多试几次，提高至少一次做对的概率。但代价是推理成本增加，且错误样本会被大量生成。
    - G 决定了你愿意为“偶然做对”支付多少采样预算。

- Db（每次采样的题目数）→ 买“题型覆盖广度”
    - Db 变大不是让单题更容易对，而是把探索预算铺到更大的题型空间里。
    - Db 管广度，G 管深度，二者形成 trade-off。

- Epoch（每轮专家数据重复训练次数）→ 买“对本轮数据的榨取程度”
    - 类似普通 SFT 的 epoch，但这里的专家数据是动态产生的，质量不均匀。
    - 太小 → 新数据利用不充分；太大 → 容易过拟合本轮采出来的偶发正确路径。

In [ ]:
def run_expert_iteration(
    model_id,
    train_data_path,
    val_data_path,
    prompt_path,
    output_dir,
    lr,
    batch_size,
    micro_batch_size,
    seed,
    max_tokens,
    n_ei_steps,
    ei_batch_size,
    rollouts,
    epochs_per_ei,
    device="cuda:0",
    vllm_device="cuda:1",
    vllm_gpu_util=0.2,
    max_eval_samples=100,
    wandb_project="cs336-ei-dynamic",
    wandb_run_name=None,
    eval_every_steps=4,
):
    config_dict = {
        "model_id": model_id,
        "train_data_path": train_data_path,
        "val_data_path": val_data_path,
        "prompt_path": prompt_path,
        "output_dir": output_dir,
        "lr": lr,
        "batch_size": batch_size,
        "micro_batch_size": micro_batch_size,
        "seed": seed,
        "max_tokens": max_tokens,
        "n_ei_steps": n_ei_steps,
        "ei_batch_size": ei_batch_size,
        "rollouts": rollouts,
        "epochs_per_ei": epochs_per_ei,
        "device": device,
        "vllm_device": vllm_device,
        "vllm_gpu_util": vllm_gpu_util,
        "max_eval_samples": max_eval_samples,
        "wandb_project": wandb_project,
        "wandb_run_name": wandb_run_name,
        "eval_every_steps": eval_every_steps,
    }

    # 1. 初始化实验
    grad_accum_steps, run, r1_template = init_ei_experiment(
        batch_size=batch_size,
        micro_batch_size=micro_batch_size,
        wandb_project=wandb_project,
        wandb_run_name=wandb_run_name,
        prompt_path=prompt_path,
        extra_config=config_dict,
    )

    # 2. 模型 / tokenizer / optimizer / vLLM
    tokenizer, policy, optimizer, vllm_inst = load_ei_models_and_optimizer(
        model_id=model_id,
        device=device,
        vllm_device=vllm_device,
        seed=seed,
        vllm_gpu_util=vllm_gpu_util,
        lr=lr,
    )
    optimizer.zero_grad(set_to_none=True)

    # 3. question pool / validation set
    question_pool = prepare_question_pool(
        train_data_path=train_data_path,
        prompt_template=r1_template,
    )
    val_samples = prepare_validation_samples(
        val_data_path=val_data_path,
        prompt_template=r1_template,
        max_eval_samples=max_eval_samples,
    )
    eval_params, rollout_params = build_sampling_params(
        max_tokens=max_tokens,
        rollouts=rollouts,
    )

    # 4. Step 0 baseline
    run_initial_ei_evaluation(
        policy=policy,
        vllm_inst=vllm_inst,
        eval_params=eval_params,
        val_samples=val_samples,
    )

    # 5. EI 主循环
    global_optim_step = 0

    for ei_step in range(n_ei_steps):
        print(f"\n{'=' * 20} 开始 EI 第 {ei_step + 1} 代演化 {'=' * 20}")

        # A. rollout
        policy.eval()
        load_policy_into_vllm_instance(policy, vllm_inst)

        batch_db, outputs = sample_rollouts_from_question_pool(
            question_pool=question_pool,
            ei_batch_size=ei_batch_size,
            vllm_inst=vllm_inst,
            rollout_params=rollout_params,
        )

        # B. filter
        expert_raw_data, success_rate = filter_expert_samples(batch_db, outputs)
        print(f">> 采样成功率: {success_rate:.2%} | 获得专家样本: {len(expert_raw_data)}")

        wandb.log({
            "ei/success_rate": success_rate,
            "ei/collected_count": len(expert_raw_data),
            "ei_step": ei_step + 1,
        })

        if len(expert_raw_data) < batch_size:
            print("!! 专家样本太少，无法凑够一个 logical batch，跳过本轮训练。")
            continue

        # C. tokenize 当前轮专家样本
        print(">> 正在对新数据进行预分词...")
        tokenized_expert_data = tokenize_expert_data(expert_raw_data, tokenizer)

        # D. 动态 train_steps
        train_steps = compute_train_steps_from_expert_data(
            num_expert_samples=len(expert_raw_data),
            epochs_per_ei=epochs_per_ei,
            batch_size=batch_size,
        )
        print(f">> 启动 SFT 训练: 执行 {train_steps} 步更新 (等效 {epochs_per_ei} Epochs)...")

        policy.train()
        pbar = tqdm(range(train_steps), desc=f"EI-{ei_step + 1} Training")

        for _ in pbar:
            global_optim_step += 1

            train_one_ei_optimizer_step(
                policy=policy,
                optimizer=optimizer,
                tokenized_expert_data=tokenized_expert_data,
                micro_batch_size=micro_batch_size,
                grad_accum_steps=grad_accum_steps,
                device=device,
                global_step=global_optim_step,
            )

            run_mid_training_eval_if_needed(
                policy=policy,
                vllm_inst=vllm_inst,
                eval_params=eval_params,
                val_samples=val_samples,
                global_optim_step=global_optim_step,
                eval_every_steps=eval_every_steps,
            )

        # E. iteration 结束评估
        run_end_of_iteration_eval(
            policy=policy,
            vllm_inst=vllm_inst,
            eval_params=eval_params,
            val_samples=val_samples,
            global_optim_step=global_optim_step,
        )

        # F. 保存 checkpoint
        save_ei_checkpoint(
            policy=policy,
            tokenizer=tokenizer,
            output_dir=output_dir,
            ei_step=ei_step + 1,
            global_optim_step=global_optim_step,
        )

        torch.cuda.empty_cache()

    print("\nExpert Iteration 任务全部完成")
    wandb.finish()

    return {
        "policy": policy,
        "tokenizer": tokenizer,
        "vllm_inst": vllm_inst,
    }

In [ ]:
run_expert_iteration(
    model_id="model/Qwen2.5-Math-1.5B",
    train_data_path="data/gsm8k/train_sft_reason_gsm8k_raw.jsonl",
    val_data_path="data/gsm8k/test.jsonl",
    prompt_path="cs336_alignment/prompts/r1_zero.prompt",
    output_dir="result/ei_checkpoints",
    lr=1e-5,
    batch_size=16,
    micro_batch_size=1,
    seed=42,
    max_tokens=1024,
    n_ei_steps=5,
    ei_batch_size=512,
    rollouts=8,
    epochs_per_ei=1,
    device="cuda:0",
    vllm_device="cuda:1",
    vllm_gpu_util=0.2,
    max_eval_samples=100,
    wandb_project="cs336-ei-dynamic",
    wandb_run_name="ei-demo-run",
    eval_every_steps=4,
)